# Policy-rate reconstruction validation

This notebook compares the reconstructed 3M/6M policy-rate proxies from ZQ Fed Funds futures with the existing Bloomberg-derived `policy_3m` and `policy_6m` series.

**Overlap used:** September 2019 through August 2023 (48 monthly observations).

Validation metrics:
- Level correlation
- MAE / RMSE / mean bias
- Regression: Bloomberg = alpha + beta × reconstructed
- R²
- Correlation of monthly changes
- Directional agreement of monthly changes
- Time-series and scatter plots
- Largest absolute errors

Important: the reconstructed series uses the simple contract mapping:
- 3M = contract month +3
- 6M = contract month +6
- implied rate = 100 − ZQ close price

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path("/mnt/data")
XLSX = BASE / "deterministic_data2 - Oct26.xlsx"
ZQ = BASE / "fed_funds_futures_history_2000_2023Aug.csv"

assert XLSX.exists(), XLSX
assert ZQ.exists(), ZQ

In [ ]:
# Load Bloomberg policy series
p3 = pd.read_excel(XLSX, sheet_name="policy_3m")
p6 = pd.read_excel(XLSX, sheet_name="policy_6m")

p3["Date"] = pd.to_datetime(p3["Date"])
p6["Date"] = pd.to_datetime(p6["Date"])
p3["decision_month"] = p3["Date"].dt.to_period("M").astype(str)
p6["decision_month"] = p6["Date"].dt.to_period("M").astype(str)

p3 = p3[["decision_month", "implied_policy_rate_3m_pct"]].rename(
    columns={"implied_policy_rate_3m_pct": "bloomberg_3m"}
)
p6 = p6[["decision_month", "implied_policy_rate_6m_pct"]].rename(
    columns={"implied_policy_rate_6m_pct": "bloomberg_6m"}
)

# Load reconstructed ZQ series
zq = pd.read_csv(ZQ)
zq3 = zq[zq["target_horizon_months"] == 3][["decision_month", "implied_rate_pct"]].rename(
    columns={"implied_rate_pct": "reconstructed_3m"}
)
zq6 = zq[zq["target_horizon_months"] == 6][["decision_month", "implied_rate_pct"]].rename(
    columns={"implied_rate_pct": "reconstructed_6m"}
)

cmp = p3.merge(p6, on="decision_month").merge(zq3, on="decision_month").merge(zq6, on="decision_month")
cmp = cmp.sort_values("decision_month").reset_index(drop=True)

print("Overlap:", cmp["decision_month"].iloc[0], "to", cmp["decision_month"].iloc[-1])
print("N =", len(cmp))
cmp.head()

In [ ]:
def validation_metrics(df, b_col, r_col):
    y = df[b_col].astype(float).to_numpy()
    x = df[r_col].astype(float).to_numpy()
    err = x - y

    corr = np.corrcoef(x, y)[0, 1]
    mae = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err**2))
    bias = np.mean(err)

    beta = np.cov(x, y, ddof=0)[0, 1] / np.var(x)
    alpha = np.mean(y) - beta * np.mean(x)
    yhat = alpha + beta * x
    r2 = 1 - np.sum((y-yhat)**2) / np.sum((y-y.mean())**2)

    dx = np.diff(x)
    dy = np.diff(y)
    change_corr = np.corrcoef(dx, dy)[0, 1]
    direction_agreement = np.mean(np.sign(dx) == np.sign(dy))

    return {
        "Correlation": corr,
        "MAE (pp)": mae,
        "RMSE (pp)": rmse,
        "Mean bias, reconstructed - Bloomberg (pp)": bias,
        "Regression alpha": alpha,
        "Regression beta": beta,
        "R-squared": r2,
        "Change correlation": change_corr,
        "Directional agreement": direction_agreement,
        "Max absolute error (pp)": np.max(np.abs(err)),
    }

summary = pd.DataFrame({
    "3M": validation_metrics(cmp, "bloomberg_3m", "reconstructed_3m"),
    "6M": validation_metrics(cmp, "bloomberg_6m", "reconstructed_6m"),
}).T

summary

In [ ]:
# Time-series comparison
fig, ax = plt.subplots(figsize=(11, 5))
x = pd.to_datetime(cmp["decision_month"])
ax.plot(x, cmp["bloomberg_3m"], label="Bloomberg 3M")
ax.plot(x, cmp["reconstructed_3m"], label="Reconstructed 3M")
ax.set_title("3M policy expectation: Bloomberg vs ZQ reconstruction")
ax.set_ylabel("Percent")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
x = pd.to_datetime(cmp["decision_month"])
ax.plot(x, cmp["bloomberg_6m"], label="Bloomberg 6M")
ax.plot(x, cmp["reconstructed_6m"], label="Reconstructed 6M")
ax.set_title("6M policy expectation: Bloomberg vs ZQ reconstruction")
ax.set_ylabel("Percent")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter + fitted regression
fig, ax = plt.subplots(figsize=(6, 6))
x = cmp["reconstructed_3m"].to_numpy()
y = cmp["bloomberg_3m"].to_numpy()
beta = np.cov(x, y, ddof=0)[0, 1] / np.var(x)
alpha = y.mean() - beta * x.mean()
grid = np.linspace(min(x.min(), y.min()), max(x.max(), y.max()), 100)
ax.scatter(x, y)
ax.plot(grid, grid, linestyle="--", label="45-degree line")
ax.plot(grid, alpha + beta * grid, label="OLS fit")
ax.set_xlabel("Reconstructed 3M (%)")
ax.set_ylabel("Bloomberg 3M (%)")
ax.set_title("3M level validation")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
x = cmp["reconstructed_6m"].to_numpy()
y = cmp["bloomberg_6m"].to_numpy()
beta = np.cov(x, y, ddof=0)[0, 1] / np.var(x)
alpha = y.mean() - beta * x.mean()
grid = np.linspace(min(x.min(), y.min()), max(x.max(), y.max()), 100)
ax.scatter(x, y)
ax.plot(grid, grid, linestyle="--", label="45-degree line")
ax.plot(grid, alpha + beta * grid, label="OLS fit")
ax.set_xlabel("Reconstructed 6M (%)")
ax.set_ylabel("Bloomberg 6M (%)")
ax.set_title("6M level validation")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Largest errors
cmp["error_3m"] = cmp["reconstructed_3m"] - cmp["bloomberg_3m"]
cmp["error_6m"] = cmp["reconstructed_6m"] - cmp["bloomberg_6m"]

print("Largest 3M errors")
display(cmp.assign(abs_error=cmp["error_3m"].abs())
        .nlargest(10, "abs_error")
        [["decision_month", "bloomberg_3m", "reconstructed_3m", "error_3m"]])

print("Largest 6M errors")
display(cmp.assign(abs_error=cmp["error_6m"].abs())
        .nlargest(10, "abs_error")
        [["decision_month", "bloomberg_6m", "reconstructed_6m", "error_6m"]])

In [ ]:
# Sensitivity: exclude September 2019, the largest outlier
cmp_ex = cmp[cmp["decision_month"] != "2019-09"].copy()

summary_ex = pd.DataFrame({
    "3M": validation_metrics(cmp_ex, "bloomberg_3m", "reconstructed_3m"),
    "6M": validation_metrics(cmp_ex, "bloomberg_6m", "reconstructed_6m"),
}).T

summary_ex

## Interpretation

A reconstruction is encouraging if:
- level correlation is very high,
- beta is close to 1,
- alpha is close to 0,
- MAE/RMSE are small in basis-point terms,
- monthly changes remain strongly correlated.

Do **not** automatically use regression-calibrated historical values in the backtest. The overlap regression is primarily diagnostic. A calibration fitted using 2019–2023 and then applied backward to 2000 would use later information in the construction. Prefer the uncalibrated, economically defined ZQ series unless the exact Bloomberg construction can be identified independently.